# 07b — Validación de la carga de datos (solo lectura)
### TFM · Análisis multidimensional del estrés estudiantil · Complemento del notebook 07

> **Cómo ejecutarlo:** menú **Kernel → Restart Kernel and Run All Cells**. No carga, modifica ni borra nada: la conexión se abre en **modo solo lectura**. Se puede ejecutar todas las veces que quieras.

El notebook 07 carga los 12 datasets en MySQL y solo puede ejecutarse una vez. Este notebook **comprueba que esa carga quedó bien**, comparando lo que hay en la base de datos con lo que imprimió el 07 al cargar. Cada comprobación termina en **OK** o **FALLA**, y al final hay un resumen.

| Bloque | Qué comprueba |
|---|---|
| 1. Objetos | Existen las 11 tablas de hechos, las 7 dimensiones y las 6 vistas de la estrella |
| 2. Filas por tabla | Cada tabla tiene exactamente las filas cargadas (suman 334.318) |
| 3. Estrella | Los 12 datasets aparecen en `vw_hechos_estres` con sus filas; 9 son fuentes con señal real (319.318 registros) |
| 4. Integridad | Sin claves naturales repetidas, ni claves hacia las dimensiones nulas o huérfanas |
| 5. Calidad | Reglas aplicadas en la ingesta: textos corruptos de Dreaddit excluidos, edades del DASS válidas |
| 6. Filtro global | La consulta «Mujeres de 18-25 con estrés Alto» reproduce los 7.446 casos de la verificación del 07 |

## Celda 1 — Conexión y valores esperados

In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text, event

DB_USER, DB_PASS = "root", ""
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")

# Garantía de solo lectura: cualquier intento de escribir en la base daría error
if engine.dialect.name == "mysql":
    @event.listens_for(engine, "connect")
    def _solo_lectura(dbapi_con, registro):
        cur = dbapi_con.cursor()
        cur.execute("SET SESSION TRANSACTION READ ONLY")
        cur.close()
try:
    with engine.connect() as con:
        con.execute(text("SELECT 1"))
except Exception as e:
    raise RuntimeError(f"No se pudo conectar a {DB_NAME}. ¿Está MySQL en verde en XAMPP? ({type(e).__name__})")
print("Conexión OK ->", DB_NAME, "(modo solo lectura: este notebook no modifica nada)")

# ── Valores esperados: los que imprimió el notebook 07 al cargar ──────────────────────────
TABLAS = {"b1_academic_stress": 140, "b1_anxiety": 13464, "b1_mental_health_survey": 87, "b1_student_habits": 1000,
          "b2_social_media_sleep": 11000, "b2_smmh": 481, "b2_lifestyle": 3000, "b3_dreaddit_post": 3549,
          "b3_depression_student": 502, "b4_dass_respondente": 39767, "b4_mental_health": 261328}
CLAVES_NATURALES = {"b1_student_habits": "id_estudiante", "b2_social_media_sleep": "id_usuario_red", "b3_dreaddit_post": "id_fragmento_post"}   # el resto de claves es autonumérica
DIMENSIONES = ["dim_bloque", "dim_dataset", "dim_genero", "dim_rango_edad", "dim_nivel_educativo", "dim_nivel_estres", "dim_subreddit"]
VISTAS = ["vw_hechos_estres", "vw_dim_dataset", "vw_dim_genero", "vw_dim_rango_edad", "vw_dim_nivel_educativo", "vw_dim_nivel_estres"]
POR_DATASET = {1: 140, 2: 13464, 3: 87, 4: 1000, 5: 11000, 6: 481, 7: 3000, 8: 2834, 9: 715, 10: 502, 11: 39767, 12: 261328}   # Dreaddit: 8 = train, 9 = test
FILTRO_GLOBAL = {11: 6799, 2: 224, 4: 183, 6: 110, 10: 65, 7: 51, 3: 14}      # Mujeres 18-25 con estrés Alto, por id_dataset
CLAVES_CONFORMES = [("id_dataset", "dim_dataset"), ("id_genero", "dim_genero"), ("id_rango_edad", "dim_rango_edad"),
                    ("id_nivel_educativo", "dim_nivel_educativo"), ("id_nivel_estres", "dim_nivel_estres")]

RESULTADOS = []
def contar(objeto):
    """Filas de una tabla o vista; None si no existe."""
    try:
        with engine.connect() as con:
            return int(con.execute(text(f"SELECT COUNT(*) FROM {objeto}")).scalar())
    except Exception:
        return None
def valor(sql):
    """Primer valor de una consulta; None si falla."""
    try:
        with engine.connect() as con:
            return con.execute(text(sql)).scalar()
    except Exception:
        return None
def consulta(sql):
    try:
        return pd.read_sql(text(sql), engine)
    except Exception:
        return None
def registrar(grupo, nombre, esperado, obtenido):
    ok = obtenido is not None and obtenido == esperado
    RESULTADOS.append({"grupo": grupo, "comprobación": nombre, "esperado": esperado, "obtenido": obtenido, "estado": "OK" if ok else "FALLA"})
    return ok
marca = lambda ok: "✔" if ok else "✘"

## 1 · Objetos de la base de datos

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
faltan = []
for o in list(TABLAS) + DIMENSIONES + VISTAS:
    existe = contar(o) is not None
    registrar("1. Objetos", o, "existe", "existe" if existe else "no existe")
    if not existe:
        faltan.append(o)
total_obj = len(TABLAS) + len(DIMENSIONES) + len(VISTAS)
if faltan:
    print(f"✘ Faltan {len(faltan)} de {total_obj} objetos: {', '.join(faltan)}")
    print("  -> modelo_datos_mysql_v2.sql crea tablas y dimensiones; esquema_estrella_powerbi.sql crea las vistas.")
else:
    print(f"✔ Existen los {total_obj} objetos: {len(TABLAS)} tablas de hechos, {len(DIMENSIONES)} dimensiones y {len(VISTAS)} vistas.")

## 2 · Filas por tabla
Mismo listado que imprimió el notebook 07 al cargar, ahora con su valor esperado.

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
print("Filas por tabla (esperado = salida del notebook 07):\n")
suma = 0
for t, esperado in TABLAS.items():
    n = contar(t)
    ok = registrar("2. Filas por tabla", t, esperado, n)
    suma += n or 0
    print(f"  {marca(ok)} {t:<26}{(f'{n:,}' if n is not None else '—'):>10} filas   (esperado {esperado:,})")
ok = registrar("2. Filas por tabla", "TOTAL de las 11 tablas", 334318, suma)
print(f"\n  {marca(ok)} TOTAL {suma:,} filas   (esperado 334,318)")
print("\n  Nota: si una tabla tiene el doble de filas, el notebook 07 se ejecutó dos veces; hay que recargar desde cero.")

## 3 · La estrella: los 12 datasets
Cada registro conserva su fuente en `id_dataset`. Dreaddit ocupa dos filas (8 = train, 9 = test) porque ambas particiones comparten tabla.

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
df = consulta("""SELECT ds.id_dataset, ds.dataset, ds.bloque, ds.uso_modelado, COUNT(*) AS filas
                 FROM vw_hechos_estres h JOIN vw_dim_dataset ds ON ds.id_dataset = h.id_dataset
                 GROUP BY ds.id_dataset, ds.dataset, ds.bloque, ds.uso_modelado ORDER BY ds.id_dataset""")
if df is None:
    registrar("3. Estrella", "consulta a vw_hechos_estres", "ok", "error")
    print("✘ No se pudo consultar la estrella: ejecuta esquema_estrella_powerbi.sql.")
else:
    df["filas"] = df["filas"].astype(int)
    df["esperado"] = df["id_dataset"].map(POR_DATASET)
    df["estado"] = [marca(f == e) for f, e in zip(df["filas"], df["esperado"])]
    display(df)
    for _, r in df.iterrows():
        registrar("3. Estrella", f"dataset {int(r.id_dataset)} · {r.dataset}", int(r.esperado), int(r.filas))
    registrar("3. Estrella", "datasets presentes en la estrella", 12, len(df))
    registrar("3. Estrella", "total de registros en vw_hechos_estres", 334318, int(df["filas"].sum()))
    reales = df[df["uso_modelado"] == "incluido"]
    registrar("3. Estrella", "fuentes con señal real ('incluido')", 9, len(reales))
    registrar("3. Estrella", "registros de las fuentes con señal real", 319318, int(reales["filas"].sum()))
    print(f"\nEstrella: {int(df['filas'].sum()):,} registros | {len(reales)} fuentes con señal real = {int(reales['filas'].sum()):,} registros "
          f"(334,318 − 15,000 de los 3 sintéticos = 319,318)")

## 4 · Integridad
Sin claves naturales repetidas (las tres tablas cuya clave viene del CSV; una carga doble las duplicaría) y sin claves hacia las dimensiones conformes que sean nulas o apunten a un registro inexistente. En las otras ocho tablas la clave es autonumérica y no puede repetirse: una carga doble se detecta en el recuento de filas del bloque 2.

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
print("Claves naturales repetidas (debe ser 0):")
for t, pk in CLAVES_NATURALES.items():
    rep = valor(f"SELECT COUNT(*) - COUNT(DISTINCT {pk}) FROM {t}")
    ok = registrar("4. Integridad", f"{t}: claves naturales repetidas", 0, int(rep) if rep is not None else None)
    print(f"  {marca(ok)} {t:<26} {int(rep) if rep is not None else '—'}")

print("\nClaves conformes en la estrella (nulas y huérfanas; debe ser 0):")
for col, dim in CLAVES_CONFORMES:
    nulas = valor(f"SELECT COUNT(*) FROM vw_hechos_estres WHERE {col} IS NULL")
    huerf = valor(f"SELECT COUNT(*) FROM vw_hechos_estres h LEFT JOIN {dim} d ON d.{col} = h.{col} WHERE d.{col} IS NULL AND h.{col} IS NOT NULL")
    ok1 = registrar("4. Integridad", f"{col}: nulas", 0, int(nulas) if nulas is not None else None)
    ok2 = registrar("4. Integridad", f"{col}: huérfanas respecto a {dim}", 0, int(huerf) if huerf is not None else None)
    print(f"  {marca(ok1 and ok2)} {col:<20} nulas: {nulas if nulas is not None else '—'} | huérfanas: {huerf if huerf is not None else '—'}")

## 5 · Calidad: reglas aplicadas durante la ingesta

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
# R7 - los fragmentos de Dreaddit con el texto corrupto '#NAME?' (error de Excel) se excluyen del corpus
corruptos = valor("SELECT COUNT(*) FROM b3_dreaddit_post WHERE texto IN ('#NAME?', '#REF!', '#VALUE!')")
ok = registrar("5. Calidad", "Dreaddit: textos corruptos (#NAME?) en la tabla", 0, int(corruptos) if corruptos is not None else None)
print(f"  {marca(ok)} Dreaddit sin textos corruptos: {corruptos}   (la fuente trae 4; se excluyeron al cargar)")

# La partición original train/test se conserva
for part, esperado in [("train", 2834), ("test", 715)]:
    n = valor(f"SELECT COUNT(*) FROM b3_dreaddit_post WHERE particion = '{part}'")
    ok = registrar("5. Calidad", f"Dreaddit: filas de la partición {part}", esperado, int(n) if n is not None else None)
    print(f"  {marca(ok)} Dreaddit {part:<5} {n if n is not None else '—'} filas   (esperado {esperado:,})")

# DASS-42: se filtraron edades inválidas (13-90) y las subescalas suman 14 ítems de 1 a 4 puntos (14-56)
emin, emax = valor("SELECT MIN(edad) FROM b4_dass_respondente"), valor("SELECT MAX(edad) FROM b4_dass_respondente")
ok = registrar("5. Calidad", "DASS: edades dentro de 13-90", True, (emin is not None and int(emin) >= 13 and int(emax) <= 90))
print(f"  {marca(ok)} DASS edades: mínima {emin}, máxima {emax}   (rango válido 13-90)")
fuera = valor("""SELECT COUNT(*) FROM b4_dass_respondente WHERE score_depresion NOT BETWEEN 14 AND 56
                 OR score_ansiedad NOT BETWEEN 14 AND 56 OR score_estres NOT BETWEEN 14 AND 56""")
ok = registrar("5. Calidad", "DASS: puntuaciones de subescala fuera de 14-56", 0, int(fuera) if fuera is not None else None)
print(f"  {marca(ok)} DASS puntuaciones fuera de 14-56: {fuera}")

## 6 · Filtro global del tutor: «Mujeres de 18-25 años con estrés Alto»
Es la consulta con la que terminaba el notebook 07: un único filtro sobre las cuatro dimensiones conformes que atraviesa varias fuentes a la vez.

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
q = consulta("""SELECT ds.id_dataset, ds.bloque, ds.dataset, COUNT(*) AS casos
                FROM vw_hechos_estres h
                JOIN vw_dim_dataset      ds ON ds.id_dataset     = h.id_dataset
                JOIN vw_dim_genero       g  ON g.id_genero       = h.id_genero
                JOIN vw_dim_rango_edad   r  ON r.id_rango_edad   = h.id_rango_edad
                JOIN vw_dim_nivel_estres e  ON e.id_nivel_estres = h.id_nivel_estres
                WHERE g.codigo = 'F' AND r.rango = '18-25' AND e.nivel = 'Alto'
                GROUP BY ds.id_dataset, ds.bloque, ds.dataset ORDER BY casos DESC""")
print("=" * 80); print("VERIFICACION: Mujeres 18-25 con estrés Alto (vía vw_hechos_estres)"); print("=" * 80)
if q is None:
    registrar("6. Filtro global", "consulta del filtro global", "ok", "error")
    print("✘ No se pudo ejecutar la consulta (faltan las vistas de la estrella).")
else:
    q["casos"] = q["casos"].astype(int)
    q["esperado"] = q["id_dataset"].map(FILTRO_GLOBAL)
    q["estado"] = [marca(c == e) for c, e in zip(q["casos"], q["esperado"])]
    display(q[["bloque", "dataset", "casos", "esperado", "estado"]])
    obtenido = dict(zip(q["id_dataset"], q["casos"]))
    for ds_id, esperado in FILTRO_GLOBAL.items():
        registrar("6. Filtro global", f"dataset {ds_id}: casos", esperado, obtenido.get(ds_id))
    registrar("6. Filtro global", "fuentes que aparecen en el filtro", 7, len(q))
    total = int(q["casos"].sum())
    ok = registrar("6. Filtro global", "total de casos", 7446, total)
    print(f"\n  {marca(ok)} TOTAL {total:,} casos en {len(q)} fuentes   (esperado 7,446 en 7 fuentes)")

## Resumen de la validación

In [ ]:
if "engine" not in globals() or "RESULTADOS" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (conexión y valores esperados). Usa Kernel > Restart Kernel and Run All Cells.")
res = pd.DataFrame(RESULTADOS)
por_grupo = res.groupby("grupo")["estado"].apply(lambda s: f"{int((s == 'OK').sum())} de {len(s)} OK").rename("resultado").to_frame()
display(por_grupo)
fallas = res[res["estado"] != "OK"]
print()
if fallas.empty:
    print(f"✅ VALIDACIÓN COMPLETADA: la carga del notebook 07 es correcta ({len(res)} de {len(res)} comprobaciones OK).")
else:
    display(fallas[["grupo", "comprobación", "esperado", "obtenido"]])
    print(f"⚠️ VALIDACIÓN CON FALLAS: {len(fallas)} de {len(res)} comprobaciones no coinciden. Revisa la tabla de arriba.")
n_items = contar("b4_dass_item")
print(f"\nInformativo: b4_dass_item (opcional, 42 filas por respondente) tiene {n_items:,} filas." if n_items else
      "\nInformativo: b4_dass_item (opcional) no se cargó, tal como está configurado por defecto en el notebook 07.")